In [1]:
import json
from pathlib import Path

import torch
import numpy as np

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p1"

In [2]:
label_map = json.loads(Path("../data/ner_label_map.json").read_text())
label2id = {k: int(v) for k, v in label_map["label2id"].items()}
id2label = {int(k): v for k, v in label_map["id2label"].items()}
LABEL_LIST = [id2label[i] for i in range(len(id2label))]
print(LABEL_LIST)


['O', 'B-ADD_ON', 'I-ADD_ON', 'B-DISH', 'I-DISH', 'B-DRINK', 'I-DRINK', 'B-MODIFIER', 'I-MODIFIER', 'B-QUANTITY', 'I-QUANTITY', 'B-REMOVE', 'I-REMOVE', 'B-SIZE', 'I-SIZE', 'B-TABLE', 'I-TABLE', 'B-TAKE_AWAY', 'I-TAKE_AWAY']


## Step 1 — Load splits

In [3]:
from datasets import Dataset

def load_split(jsonl_path: str) -> Dataset:
    records = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records.append({
                "input_ids": rec["input_ids"],
                "attention_mask": rec["attention_mask"],
                "labels": rec["labels"],
            })
    return Dataset.from_list(records)


train_dataset = load_split("../data/ner_dataset_train.jsonl")
val_dataset   = load_split("../data/ner_dataset_val.jsonl")
test_dataset  = load_split("../data/ner_dataset_test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1600
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


## Step 2 — Class imbalance

Entity tags skew toward `O` and toward common tags (DISH, QUANTITY) vs rare ones (REMOVE, TABLE). Inverse-frequency class weights computed from the **train split only**, same convention as `notebooks/3.2`'s Step 4.

In [4]:
from collections import Counter

def compute_label_counts(jsonl_path: str) -> Counter:
    """Count label ids across a split, excluding -100 (ignored/padding)."""
    counts = Counter()
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for lbl_id in rec["labels"]:
                if lbl_id != -100:
                    counts[lbl_id] += 1
    return counts


train_label_counts = compute_label_counts("../data/ner_dataset_train.jsonl")
for lbl_id in range(len(LABEL_LIST)):
    print(f"  {id2label[lbl_id]:14s}: {train_label_counts[lbl_id]:6d}")


  O             :  15103
  B-ADD_ON      :     91
  I-ADD_ON      :     27
  B-DISH        :   1408
  I-DISH        :   1871
  B-DRINK       :    832
  I-DRINK       :   1069
  B-MODIFIER    :    333
  I-MODIFIER    :    213
  B-QUANTITY    :   1419
  I-QUANTITY    :    638
  B-REMOVE      :    425
  I-REMOVE      :    260
  B-SIZE        :    212
  I-SIZE        :    172
  B-TABLE       :     10
  I-TABLE       :     11
  B-TAKE_AWAY   :     62
  I-TAKE_AWAY   :     37


In [5]:
total_tokens = sum(train_label_counts.values())
n_classes = len(LABEL_LIST)

class_weights = torch.tensor(
    [total_tokens / (n_classes * max(train_label_counts[i], 1)) for i in range(n_classes)],
    dtype=torch.float,
)

for lbl_id, weight in enumerate(class_weights.tolist()):
    print(f"  {id2label[lbl_id]:14s}: weight={weight:.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)


  O             : weight=0.084
  B-ADD_ON      : weight=13.992
  I-ADD_ON      : weight=47.160
  B-DISH        : weight=0.904
  I-DISH        : weight=0.681
  B-DRINK       : weight=1.530
  I-DRINK       : weight=1.191
  B-MODIFIER    : weight=3.824
  I-MODIFIER    : weight=5.978
  B-QUANTITY    : weight=0.897
  I-QUANTITY    : weight=1.996
  B-REMOVE      : weight=2.996
  I-REMOVE      : weight=4.897
  B-SIZE        : weight=6.006
  I-SIZE        : weight=7.403
  B-TABLE       : weight=127.332
  I-TABLE       : weight=115.756
  B-TAKE_AWAY   : weight=20.537
  I-TAKE_AWAY   : weight=34.414


### Sanity check

In [6]:
assert class_weights.argmin().item() == label2id["O"], "O should get the lowest weight"
assert loss_fn.ignore_index == -100
print("Step 2 sanity checks passed.")


Step 2 sanity checks passed.


## Step 3 — Model architecture

`BertCrfForTokenClassification` (`../src/ordo_ai/models/ner_crf.py`): full fine-tune of `indobenchmark/indobert-base-p1` encoder + linear emissions + a CRF decoding layer. The CRF's learned transition matrix forbids invalid BIO transitions (e.g. `I-X` with no preceding `B-X`) that plain per-token argmax cannot prevent.

In [7]:
import sys
sys.path.insert(0, "../src")
from ordo_ai.models.ner_crf import BertCrfForTokenClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = BertCrfForTokenClassification.from_pretrained_bert(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")


[transformers] You passed `num_labels=19` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device          : mps
total params    : 124,456,354
trainable params: 124,456,354  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [8]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)

logits = outputs["logits"]
print(f"logits shape: {tuple(logits.shape)}  (expect (4, {MAX_LENGTH}, {len(LABEL_LIST)}))")
assert logits.shape == (4, MAX_LENGTH, len(LABEL_LIST))

print(f"sample CRF negative log-likelihood loss: {outputs['loss'].item():.4f}")

model.train()
print("Step 3 sanity checks passed.")


logits shape: (4, 64, 19)  (expect (4, 64, 19))
sample CRF negative log-likelihood loss: 54.4296
Step 3 sanity checks passed.


## Step 4 — Training

`CrfTrainer` overrides `compute_loss` to use the CRF negative-log-likelihood loss (Step 3) instead of `Trainer`'s per-token cross-entropy default, and overrides `prediction_step` to Viterbi-decode through the CRF layer for eval/predict instead of per-token argmax.

In [9]:
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """
    Convert per-token Viterbi-decoded predictions/labels back into BIO tag
    sequences (dropping -100 positions) and score with seqeval's
    entity-level precision/recall/F1.
    """
    preds, labels = eval_pred

    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq = [id2label[l] for l in label_row if l != -100]
        pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)

    results = seqeval_metric.compute(predictions=pred_seqs, references=true_seqs)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [10]:
from ordo_ai.tracking import init_experiment

init_experiment("menu-ner")

2026/07/01 08:36:34 INFO mlflow.tracking.fluent: Experiment with name 'menu-ner' does not exist. Creating a new experiment.


In [11]:
from transformers import Trainer, TrainingArguments

class CrfTrainer(Trainer):
    """Trainer using the model's built-in CRF negative-log-likelihood loss
    (Step 3) instead of `Trainer`'s default per-token cross-entropy. Eval/predict
    steps Viterbi-decode through the CRF layer instead of taking per-token argmax,
    since only Viterbi enforces valid BIO transitions at inference time too."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels")

        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs["loss"].detach() if labels is not None else None
            decoded = model.decode(inputs["input_ids"], inputs["attention_mask"])

        seq_len = inputs["input_ids"].shape[1]
        preds = torch.zeros((len(decoded), seq_len), dtype=torch.long)
        for i, tag_seq in enumerate(decoded):
            preds[i, : len(tag_seq)] = torch.tensor(tag_seq, dtype=torch.long)

        if prediction_loss_only:
            return (loss, None, None)
        return (loss, preds, labels)


training_args = TrainingArguments(
    output_dir="../models/indobert-ner-bio",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="epoch",
    report_to=["mlflow"],
    seed=SEED,
)

trainer = CrfTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
import mlflow

run = mlflow.start_run()
train_result = trainer.train()
train_result.metrics


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,16.525913,7.714957,0.595652,0.683860,0.636716,0.849261
2,6.676008,7.255098,0.610000,0.710483,0.656418,0.854516
3,5.695165,7.029332,0.646154,0.698835,0.671463,0.860755
4,4.975872,7.069617,0.635821,0.708819,0.670338,0.864368
5,4.496968,7.028365,0.650075,0.717138,0.681962,0.865025


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start,

{'train_runtime': 254.6891,
 'train_samples_per_second': 31.411,
 'train_steps_per_second': 1.963,
 'total_flos': 263152441344000.0,
 'train_loss': 7.6739851684570315,
 'epoch': 5.0}

### Save the fine-tuned model

In [14]:
from transformers import AutoTokenizer

final_model_path = "../models/indobert-ner-bio-final"
trainer.save_model(final_model_path)
AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")


mlflow.transformers.log_model(
    transformers_model=final_model_path,
    name="model",
    task="token-classification",
)
mlflow.end_run()

Saved fine-tuned model -> ../models/indobert-ner-bio-final


2026/07/01 08:42:18 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/indobert-ner-bio-final'. Use `repo_type` argument if needed.
2026/07/01 08:42:18 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/07/01 08:42:19 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into memory, we don't infer the pip requirement for the model. Instead, we will use the default requirements, but it may not capture all required pip libraries for the model. Consider providing the pip requirements explicitly.


🏃 View run peaceful-dog-467 at: http://localhost:5001/#/experiments/6/runs/eae20bbbeb19486f99aabb3e266c9a0f
🧪 View experiment at: http://localhost:5001/#/experiments/6


## Step 5 — Evaluation

Run on the held-out **test** split — not seen during training or `load_best_model_at_end` checkpoint selection — and report per-entity-type precision/recall/F1, not just overall, since rare tags (REMOVE, TABLE) are where this model is most likely to be weak.

In [15]:
test_predictions = trainer.predict(test_dataset)
test_preds, test_label_ids = test_predictions.predictions, test_predictions.label_ids

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


overall test metrics:
  test_loss                : 6.528270244598389
  test_precision           : 0.6231003039513677
  test_recall              : 0.7056798623063684
  test_f1                  : 0.6618240516545602
  test_accuracy            : 0.8611575777852124
  test_runtime             : 3.1808
  test_samples_per_second  : 62.877
  test_steps_per_second    : 4.087


In [16]:
test_true_seqs, test_pred_seqs = [], []
for pred_row, label_row in zip(test_preds, test_label_ids):
    true_seq = [id2label[l] for l in label_row if l != -100]
    pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    test_true_seqs.append(true_seq)
    test_pred_seqs.append(pred_seq)

per_entity_report = seqeval_metric.compute(
    predictions=test_pred_seqs, references=test_true_seqs
)

print("Per-entity-type breakdown (test set):")
for entity_type, scores in per_entity_report.items():
    if entity_type.startswith("overall"):
        continue
    print(
        f"  {entity_type:10s}  precision={scores['precision']:.3f}  "
        f"recall={scores['recall']:.3f}  f1={scores['f1']:.3f}  "
        f"n={scores['number']}"
    )
print()
print(f"  overall precision={per_entity_report['overall_precision']:.3f}  "
      f"recall={per_entity_report['overall_recall']:.3f}  "
      f"f1={per_entity_report['overall_f1']:.3f}  "
      f"accuracy={per_entity_report['overall_accuracy']:.3f}")


Per-entity-type breakdown (test set):
  ADD_ON      precision=0.667  recall=0.105  f1=0.182  n=19
  DISH        precision=0.651  recall=0.738  f1=0.692  n=172
  DRINK       precision=0.806  recall=0.897  f1=0.849  n=97
  MODIFIER    precision=0.429  recall=0.500  f1=0.462  n=30
  QUANTITY    precision=0.623  recall=0.762  f1=0.686  n=189
  REMOVE      precision=0.444  recall=0.571  f1=0.500  n=42
  SIZE        precision=0.333  recall=0.310  f1=0.321  n=29
  TABLE       precision=0.000  recall=0.000  f1=0.000  n=1
  TAKE_AWAY   precision=0.400  recall=1.000  f1=0.571  n=2

  overall precision=0.623  recall=0.706  f1=0.662  accuracy=0.861


### Misclassified examples

In [17]:
bio_records_by_id = {}
with open("../data/ner_dataset_bio.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        bio_records_by_id[rec["id"]] = rec

test_ids = []
with open("../data/ner_dataset_test.jsonl", encoding="utf-8") as f:
    for line in f:
        test_ids.append(json.loads(line)["id"])

n_mismatch = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq != pred_seq:
        n_mismatch += 1
print(f"Mismatched records: {n_mismatch} / {len(test_ids)}\n")

shown = 0
for rec_id, true_seq, pred_seq in zip(test_ids, test_true_seqs, test_pred_seqs):
    if true_seq == pred_seq or shown >= 10:
        continue
    tokens = bio_records_by_id[rec_id]["tokens"]
    print(f"id={rec_id}")
    for tok, t, p in zip(tokens, true_seq, pred_seq):
        marker = "" if t == p else "  <-- MISMATCH"
        print(f"    {tok:15s} true={t:10s} pred={p:10s}{marker}")
    print()
    shown += 1


Mismatched records: 137 / 200

id=1931
    tolong          true=O          pred=O         
    bukan           true=O          pred=O         
    bukan           true=O          pred=B-REMOVE    <-- MISMATCH
    nasi            true=O          pred=B-DISH      <-- MISMATCH
    nasi            true=O          pred=I-DISH      <-- MISMATCH
    goreng          true=O          pred=I-DISH      <-- MISMATCH
    gila            true=O          pred=I-DISH      <-- MISMATCH
    saya            true=O          pred=O         
    maunya          true=O          pred=O         
    gudeg           true=B-DISH     pred=B-DISH    
    dua             true=B-QUANTITY pred=B-QUANTITY
    sih             true=O          pred=O         
    mbak            true=O          pred=O         

id=1767
    oh              true=O          pred=O         
    iya             true=O          pred=O         
    bukan           true=O          pred=O         
    ee              true=O          pred=O        

## Step 6 — Inference

Load the saved model fresh from disk to confirm it's reloadable standalone. `predict_entities(text)` runs raw text through tokenize -> predict -> decode-to-words, then extracts contiguous `B-X`/`I-X` spans per entity type — the structured-order output downstream LangGraph nodes will consume.

In [18]:
from transformers import AutoTokenizer as _AT

INFERENCE_MODEL_PATH = "../models/indobert-ner-bio-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = BertCrfForTokenClassification.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")


Loaded model from ../models/indobert-ner-bio-final
id2label: {0: 'O', 1: 'B-ADD_ON', 2: 'I-ADD_ON', 3: 'B-DISH', 4: 'I-DISH', 5: 'B-DRINK', 6: 'I-DRINK', 7: 'B-MODIFIER', 8: 'I-MODIFIER', 9: 'B-QUANTITY', 10: 'I-QUANTITY', 11: 'B-REMOVE', 12: 'I-REMOVE', 13: 'B-SIZE', 14: 'I-SIZE', 15: 'B-TABLE', 16: 'I-TABLE', 17: 'B-TAKE_AWAY', 18: 'I-TAKE_AWAY'}


In [19]:
def predict_entities(text: str) -> dict:
    """
    Run raw text through the fine-tuned entity tagger.

    Returns:
      tokens : word-level tokens (text.split())
      labels : predicted BIO tag per word (first-subword label only)
      spans  : list of {label: <TAG>, text: str, start: int, end: int}
               (end is exclusive, word indices)
    """
    tokens = text.lower().split()
    if not tokens:
        return {"tokens": [], "labels": [], "spans": []}

    encoding = inference_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoding.word_ids()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        pred_ids = inference_model.decode(encoding["input_ids"], encoding["attention_mask"])[0]

    word_labels = [None] * len(tokens)
    prev_word_id = None
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id != prev_word_id:
            word_labels[word_id] = id2label[pred_ids[tok_idx]]
        prev_word_id = word_id
    word_labels = [lbl if lbl is not None else "O" for lbl in word_labels]

    spans = []
    i = 0
    while i < len(tokens):
        lbl = word_labels[i]
        if lbl.startswith("B-"):
            span_type = lbl[2:]
            j = i + 1
            while j < len(tokens) and word_labels[j] == f"I-{span_type}":
                j += 1
            spans.append({
                "label": span_type,
                "text": " ".join(tokens[i:j]),
                "start": i,
                "end": j,
            })
            i = j
        else:
            i += 1

    return {"tokens": tokens, "labels": word_labels, "spans": spans}


### Try it on fresh, unseen sentences

In [20]:
demo_sentences = [
    "saya mau pesan ayam bakar dua porsi pedas banget",
    "minta es teh dua gelas ya bang, jangan pakai gula",
    "tolong nasi goreng satu buat meja lima, dibungkus",
    "add sambal sama extra cheese untuk pasta-nya",
]

for text in demo_sentences:
    result = predict_entities(text)
    print(f"input  : {text}")
    print(f"tagged : {' '.join(f'{t}[{l}]' if l != 'O' else t for t, l in zip(result['tokens'], result['labels']))}")
    print(f"spans  : {result['spans']}")
    print()


input  : saya mau pesan ayam bakar dua porsi pedas banget
tagged : saya mau pesan ayam[B-DISH] bakar[I-DISH] dua[B-QUANTITY] porsi[I-QUANTITY] pedas[B-MODIFIER] banget[I-MODIFIER]
spans  : [{'label': 'DISH', 'text': 'ayam bakar', 'start': 3, 'end': 5}, {'label': 'QUANTITY', 'text': 'dua porsi', 'start': 5, 'end': 7}, {'label': 'MODIFIER', 'text': 'pedas banget', 'start': 7, 'end': 9}]

input  : minta es teh dua gelas ya bang, jangan pakai gula
tagged : minta es[B-DRINK] teh[I-DRINK] dua[B-QUANTITY] gelas[I-QUANTITY] ya bang, jangan pakai gula
spans  : [{'label': 'DRINK', 'text': 'es teh', 'start': 1, 'end': 3}, {'label': 'QUANTITY', 'text': 'dua gelas', 'start': 3, 'end': 5}]

input  : tolong nasi goreng satu buat meja lima, dibungkus
tagged : tolong nasi[B-DISH] goreng[I-DISH] satu[B-QUANTITY] buat meja lima,[B-QUANTITY] dibungkus[B-TAKE_AWAY]
spans  : [{'label': 'DISH', 'text': 'nasi goreng', 'start': 1, 'end': 3}, {'label': 'QUANTITY', 'text': 'satu', 'start': 3, 'end': 4}, {'label'